# RAG Pipeline for Financial Document Q&A

A retrieval-augmented generation system built from scratch
for querying Apple's 10-K SEC filings.

## Architecture
1. **Chunking** — sentence-aware splitting at ~1000 chars
2. **Embedding** — OpenAI text-embedding-3-small (1536 dims)
3. **Indexing** — FAISS with L2 distance (equivalent to cosine
   on normalized vectors)
4. **Retrieval** — top-k nearest neighbor search
5. **Reranking** — Cohere cross-encoder reranker for precision
6. **Generation** — GPT-4o-mini with retrieved context

## Why reranking?
FAISS retrieves by semantic similarity, it finds topically
related chunks. The Cohere reranker reads query and chunk
together and scores whether the chunk actually answers the
question. For multi-part or specific queries, this improves
answer quality meaningfully.

In [1]:
import os
from google.colab import userdata

# API keys loaded from Colab secrets - never hardcode these
try:
    OPENAI_API_KEY = userdata.get('SAClass')
    COHERE_API_KEY = userdata.get('COHERE_KEY')
except:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    COHERE_API_KEY = os.getenv("COHERE_API_KEY")

### First installing required packages

sec-edgar-downloader is a python package for downloading files from edgar

In [2]:
!pip install sec-edgar-downloader openai faiss-cpu cohere

## Downloading the latest AAPL 10-K report from the SEC Edgar database
* Using the examples from the link below to get the 10-K report
* (Note: For company name and email, using Columbia University and my Columbia University email)
* The report will be downloaded in the sec-edgar-filings folder
* [sec_edgar_downloader](https://sec-edgar-downloader.readthedocs.io/en/latest/)

In [3]:
from sec_edgar_downloader import Downloader

In [4]:
dl = Downloader("Columbia University", "your@email.com")
dl.get("10-K", "AAPL", limit=1, download_details=True)

1

## Parsing with BeautifulSoup

* The report we got is in html format and we need to parse it. BeautifulSoup comes in handy.
* The function preprecess_10k_files is templated below does exactly that.

* [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)


In [5]:
import os
from bs4 import BeautifulSoup

def preprocess_10k_files(download_dir):
    text_data = []
    for root, dirs, files in os.walk(download_dir):
        for file in files:
            if file.endswith(".html"):
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    html_doc = f.read()
                    soup = BeautifulSoup(html_doc, 'html.parser')
                    text_data.append(soup.get_text())
    return text_data

# Loading and preprocessing 10-K reports
download_dir = "sec-edgar-filings" # Using the download directory name
text_data = preprocess_10k_files(download_dir)

# Making the RAG
* text_data contains the text of the document
* creating a RAG that can help answer questions about the 10-K report

In [ ]:
import numpy as np
import faiss
from openai import OpenAI
import os
import textwrap
import cohere

# Initializing API clients if not already done
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('SAClass')
    COHERE_API_KEY = userdata.get('COHERE_KEY')
except:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    COHERE_API_KEY = os.getenv("COHERE_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
co = cohere.Client(COHERE_API_KEY)

def rerank(query, candidate_chunks, top_n=3):
    """Rerank retrieved chunks using Cohere cross-encoder."""
    results = co.rerank(
        query=query,
        documents=candidate_chunks,
        top_n=top_n,
        model="rerank-english-v3.0"
    )
    return [candidate_chunks[r.index] for r in results.results]

def chunk_text(text, chunk_size=1000):
    """Split text into chunks by sentences."""
    sentences = text.replace('. ', '.|').split('|')
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding from OpenAI."""
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

def get_all_embeddings(text_data,model="text-embedding-3-small"):
    """Get embeddings for all chunks."""
    embeddings = []
    for chunk in chunk_text(text_data):
        embedding = get_embedding(chunk,model)
        embeddings.append(embedding)
    return embeddings

def build_faiss_index(documents,model="text-embedding-3-small"):
  """Build FAISS index from text documents."""
  embeddings = get_all_embeddings(documents,model)

  #converting to numpy array
  embeddings_array = np.array(embeddings).astype('float32')

  #creating faiss index
  dimension = embeddings_array.shape[1] # Will return the number of dimensions of the embeddings
  index = faiss.IndexFlatL2(dimension)
  index.add(embeddings_array)
  return index,embeddings_array

def search(query, index, chunks, k=3):
    """Search for most relevant chunks."""
    # Getting query embedding
    query_embedding = np.array([get_embedding(query)]).astype('float32')

    # Searching in FAISS index
    distances, indices = index.search(query_embedding, k)

    # Returning results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk': chunks[idx],
            'distance': distances[0][i],
            'index': idx
        })

    return results

def construct_prompt_with_context(query, index, chunks, k=3):
    """Construct prompt with retrieved chunks."""
    results = search(query, index, chunks, k)
    context = "\n\n".join([f"Context {i+1}: {result['chunk']}" for i, result in enumerate(results)])
    prompt = f"""Based on the following context, please answer the question.
    { context}
    Question: {query}
    Answer:"""
    return prompt

def get_answer(prompt):
    """Get answer from Open AI."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
        {"role": "system", "content": "You are a helpful customer service assistant. Answer questions based only on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=500
    )

    return response.choices[0].message.content

def rag(query, index, chunks, k=3):
    """Complete RAG pipeline."""
    prompt = construct_prompt_with_context(query, index, chunks, k)
    answer = get_answer(prompt)
    return answer

def rag_with_rerank(query, index, chunks, k=10, top_n=3):
    """RAG pipeline with Cohere reranking step."""
    # Step 1: Retrieve k=10 candidates from FAISS (wider net)
    results = search(query, index, chunks, k=k)
    candidate_chunks = [r['chunk'] for r in results]

    # Step 2: Rerank candidates, keep top 3
    reranked_chunks = rerank(query, candidate_chunks, top_n=top_n)

    # Step 3: Build prompt with reranked context
    context = "\n\n".join([
        f"Context {i+1}: {chunk}"
        for i, chunk in enumerate(reranked_chunks)
    ])
    prompt = f"""Based on the following context, answer the question.
    {context}
    Question: {query}
    Answer:"""

    return get_answer(prompt)

# Preparing Document Chunks and Building FAISS Index
# Joining the list of text data into a single string document
full_text_document = " ".join(text_data)
chunks = chunk_text(full_text_document)
index, embeddings = build_faiss_index(full_text_document)

# Defining the interactive main function
def main():
    full_text_document = " ".join(text_data)
    chunks = chunk_text(full_text_document)
    index, embeddings = build_faiss_index(full_text_document)

    print("Welcome to the RAG-powered 10-K Assistant!")
    print("Ask me anything about Apple's 10-K filings.")

    while True:
        query = input("\nYour question: ").strip()
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        if not query:
            print("Please enter a question.")
            continue

        try:
            # Standard RAG pipeline
            answer = rag(query, index, chunks, k=3)
            print("\nWithout reranking:")
            print(textwrap.fill(answer, 80))

            # Reranked pipeline
            answer_reranked = rag_with_rerank(query, index, chunks)
            print("\nWith reranking:")
            print(textwrap.fill(answer_reranked, 80))

        except Exception as e:
            print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

Welcome to the RAG-powered 10-K Assistant!
Ask me anything about Apple's 10-K filings.

Your question: What was Apple's total revenue in 2023?

Without reranking:
Apple's total revenue in 2023 was $383,285 million.

With reranking:
Apple's total revenue in 2023 was $383,285 million.

Your question: What did Apple's management say about revenue decline in China and what was their recovery outlook?

Without reranking:
Apple's management indicated that Greater China net sales decreased during 2025
compared to 2024 primarily due to lower net sales of iPhone. However, this
decline was partially offset by higher net sales of Mac. The provided context
does not mention a specific recovery outlook for China.

With reranking:
Apple's management indicated that Greater China net sales decreased during 2025
compared to 2024 primarily due to lower net sales of iPhone, which was partially
offset by higher net sales of Mac. However, the provided context does not
include any specific recovery outlook f

## What I'd improve next
- Semantic chunking to detect topic boundaries automatically
- Chunk overlap (1-2 sentences) to avoid boundary misses
- Metadata filtering by section (Item 7, Item 8) for
  targeted retrieval
- Eval framework to measure retrieval quality systematically
- Multi-document corpus (multiple company 10-Ks across years)